# 카페 매출 회귀·상관 분석 — 매출에 가장 큰 영향을 주는 변수는?

지난 전처리 실습과 **같은 데이터**(`cafe_sales.csv`)를 이번엔 판다스·시본·statsmodels로 분석합니다.

흐름: 로드·정제 → 탐색 → 상관계수 히트맵 → 다중 회귀(OLS) → 인사이트 정리

**최종 미션** — 아래 결과지를 채우세요:

> [ 카페 매출 영향 요인 분석 결과 ]
> 1. 가장 강한 상관 변수: ______ (r = ____)
> 2. 회귀 모델 설명력 (R²): ______
> 3. 통계적으로 유의한 변수: ① ______ (p < 0.05, 계수 ____) ② ______ (p < 0.05, 계수 ____)
> 4. 비즈니스 제안: ______

In [ ]:
# Step 1. 데이터 로드 및 정제
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

df = pd.read_csv("data/cafe_sales.csv")

# ⚠️ dropna()를 그대로 쓰면 분석에 안 쓰는 컬럼의 결측 때문에도 행이 삭제됩니다.
#    분석에 필요한 컬럼만 subset으로 지정하는 것이 좋습니다.
df = df.dropna(subset=["temp", "customer_cnt", "sales"])

# 지난 회차에서 배운 이상치 처리 — 안 하면 9,900만 매출이 상관·회귀를 왜곡합니다!
df = df[(df["sales"] > 0) & (df["temp"].between(-30, 45))]
q1, q3 = df["sales"].quantile([0.25, 0.75])
iqr = q3 - q1
df = df[df["sales"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]

# 주말 여부 파생 변수 생성
df["is_weekend"] = df["weekday"].isin(["Sat", "Sun"]).astype(int)

print(f"분석 대상: {len(df):,}행")
df.head()

In [ ]:
# Step 1-2. 데이터 탐색 — 분포와 이상치를 눈으로 확인
print(df[["temp", "customer_cnt", "sales"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df["sales"].hist(bins=30, ax=axes[0])
axes[0].set_title("매출 분포 (히스토그램)")
df.boxplot(column="sales", by="is_weekend", ax=axes[1])
axes[1].set_title("주중(0) vs 주말(1) 매출")
plt.suptitle("")
plt.tight_layout(); plt.show()

In [ ]:
# Step 2. 상관계수 히트맵 — 어느 변수가 매출과 함께 움직이는가?
corr = df[["temp", "customer_cnt", "is_weekend", "sales"]].corr()

plt.figure(figsize=(6, 4.5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("상관계수 히트맵")
plt.show()

print("sales와의 상관계수(절댓값 순):")
print(corr["sales"].drop("sales").abs().sort_values(ascending=False))
# 👉 결과지 1번: 가장 강한 상관 변수와 r 값을 기록하세요

In [ ]:
# Step 3. 다중 회귀 (OLS)
#
# sm.add_constant가 필요한 이유: OLS는 y = b1*x1 + b2*x2 + ... 형태만 계산하는데,
# 절편(상수항, b0)까지 구하려면 "항상 1인 가상의 컬럼"을 하나 추가해줘야 합니다.
# 이 컬럼이 없으면 회귀선이 원점(0,0)을 강제로 지나가야 한다는 잘못된 가정이 생깁니다.
X = sm.add_constant(df[["temp", "customer_cnt", "is_weekend"]])
y = df["sales"]
model = sm.OLS(y, X).fit()
print(model.summary())
# 👉 결과지 2번: R-squared 값
# 👉 결과지 3번: P>|t| 가 0.05보다 작은 변수와 그 coef
#    (P-value가 크다는 건 "이 변수의 효과가 우연일 가능성을 배제 못 한다"는 뜻입니다)

In [ ]:
# Step 4. 인사이트 정리 — summary 표를 읽기 쉽게 추출
result = pd.DataFrame({
    "계수(coef)": model.params.round(1),
    "p-value": model.pvalues.round(4),
    "유의함(p<0.05)": model.pvalues < 0.05,
})
print(result)
print(f"\nR² = {model.rsquared:.3f}  (모델이 매출 변동의 {model.rsquared*100:.0f}%를 설명)")

# 해석 가이드:
# - customer_cnt 계수 = "고객 1명이 늘 때 매출이 평균 몇 원 증가하는가"
# - is_weekend 계수 = "주말이면 매출이 평균 몇 원 더 높은가" (다른 조건 동일 시)
# - 상관이 높아도 p-value가 크면 '우연일 가능성'을 배제 못 함

## 결과지 작성 (직접 채우세요)

[ 카페 매출 영향 요인 분석 결과 ]
1. 가장 강한 상관 변수: __________ (r = ____)
2. 회귀 모델 설명력 (R²): ______
3. 통계적으로 유의한 변수:
   - ① __________ (p < 0.05, 계수 ____)
   - ② __________ (p < 0.05, 계수 ____)
4. 비즈니스 제안 (분석 결과를 실행 가능한 문장으로!):
   - 예시 형식: "____가 매출에 가장 큰 영향을 주므로, ____를 하면 매출 개선을 기대할 수 있다"

### 생각해 볼 질문
1. Step 1의 이상치 제거를 **건너뛰고** 히트맵을 다시 그려보세요. 상관계수가 어떻게 달라지나요? (전처리가 왜 분석보다 먼저인지의 증거)
2. customer_cnt의 상관이 가장 높다면, "고객 수를 늘리면 매출이 오른다"고 결론 내려도 될까요? (상관 ≠ 인과)
3. temp의 p-value가 0.05보다 크다면, "기온은 매출과 무관하다"고 말할 수 있을까요?
4. R²가 0.9라면 좋은 모델일까요? 이 데이터에서 customer_cnt가 sales를 잘 설명하는 것이 당연한 이유는?